# 의정이슈레이더 — 검색 실험 3종 및 예측 평가

이 Notebook은 담당 실험 범위를 한 파일에서 재현·설명하기 위해 구성했습니다.

1. **Raw vs Enriched** 임베딩 입력 비교
2. **Arctic vs KURE** 임베딩 모델 비교
3. **ChromaDB vs PostgreSQL+pgvector** 벡터 DB 비교
4. **예측 평가**: Rolling Backtest, F1, MAE, RMSE, 방향성 정확도

> 기본값은 이미 수행한 전체 실험 결과를 불러와 빠르게 검토하는 모드입니다. 대용량 임베딩·DB 벤치마크·예측 재학습은 아래 플래그를 `True`로 바꾼 경우에만 실행합니다.

## 0. 실험 환경과 재현 원칙

- 검색 평가는 동일한 질의·후보군·라벨·Top-K 조건으로 비교합니다.
- DB 비교는 동일한 901,358개 벡터와 동일 질의를 사용합니다.
- 예측 평가는 미래 정보 누출을 막기 위해 시간순 Rolling Backtest를 사용합니다.
- Notebook 자체는 수 초 안에 열리도록 저장 결과 확인을 기본값으로 둡니다.

In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

RUN_HEAVY_EXPERIMENTS = False   # 임베딩/DB 비교 원본 스크립트 재실행
RUN_FORECAST_RETRAIN = False    # 다중 이슈 예측 모델 재학습

def find_project_root():
    env_root = os.getenv('COUNCIL_PROJECT_ROOT')
    candidates = [
        Path(env_root) if env_root else None,
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if candidate and (candidate / 'src' / '10_compare_embedding_variants.py').exists():
            return candidate.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
print('PROJECT_ROOT:', PROJECT_ROOT)
print('실행 모드:', '원본 실험 재실행' if RUN_HEAVY_EXPERIMENTS else '저장 결과 확인')

## 1. 검색 평가 지표 구현

- **Precision@K**: 상위 K개 중 관련 문서의 비율
- **Recall@K**: 전체 관련 문서 중 상위 K개에서 찾은 비율
- **MRR**: 첫 번째 관련 문서 순위의 역수 평균
- **nDCG@K**: 관련도가 높은 문서를 상단에 배치했는지 평가하며, 이상적인 정렬 대비 0~1로 정규화
- **95% Bootstrap CI**: 질의별 성능 차이를 복원추출해 평균 차이의 불확실성을 추정

In [ ]:
def precision_at_k(relevance, k=10):
    values = np.asarray(relevance[:k], dtype=float)
    return float((values > 0).mean()) if len(values) else 0.0

def recall_at_k(relevance, total_relevant, k=10):
    if total_relevant <= 0:
        return 0.0
    return float((np.asarray(relevance[:k]) > 0).sum() / total_relevant)

def reciprocal_rank(relevance):
    for rank, value in enumerate(relevance, start=1):
        if value > 0:
            return 1.0 / rank
    return 0.0

def dcg_at_k(relevance, k=10):
    values = np.asarray(relevance[:k], dtype=float)
    if len(values) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(values) + 2))
    return float(np.sum((2 ** values - 1) / discounts))

def ndcg_at_k(relevance, k=10):
    actual = dcg_at_k(relevance, k)
    ideal = dcg_at_k(sorted(relevance, reverse=True), k)
    return actual / ideal if ideal > 0 else 0.0

def bootstrap_mean_difference(a, b, repeats=5000, seed=42):
    a, b = np.asarray(a, float), np.asarray(b, float)
    assert len(a) == len(b), '질의별 결과 길이가 같아야 합니다.'
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(repeats):
        index = rng.integers(0, len(a), size=len(a))
        diffs.append(float(np.mean(a[index] - b[index])))
    return np.quantile(diffs, [0.025, 0.975]).tolist()

# 간단한 동작 확인
sample_relevance = [2, 0, 1, 0, 0]
print({
    'P@3': precision_at_k(sample_relevance, 3),
    'MRR': reciprocal_rank(sample_relevance),
    'nDCG@3': round(ndcg_at_k(sample_relevance, 3), 4),
})

## 2. Raw vs Enriched 임베딩 입력 비교

- **Raw**: 실제 발언 본문 중심으로 임베딩
- **Enriched**: 본문에 정책분야·핵심이슈·키워드·요약 등 LLM 메타데이터를 결합해 임베딩

메타데이터를 더 넣는다고 항상 검색 성능이 좋아지는 것은 아닙니다. 질의와 직접 관련 없는 요약·분류 정보가 본문의 의미 벡터를 흐릴 수 있으므로 동일 평가셋으로 확인했습니다.

In [ ]:
raw_enriched = pd.DataFrame([
    {'variant': 'raw', 'nDCG@10': 0.8928, 'Precision@10': 0.9750, 'Recall@20': 0.5826, 'MRR': 1.0000},
    {'variant': 'enriched', 'nDCG@10': 0.8871, 'Precision@10': 0.9667, 'Recall@20': 0.5744, 'MRR': 0.9896},
]).set_index('variant')

raw_enriched_ci = [-0.0224, 0.0110]  # enriched - raw nDCG@10의 95% CI
display(raw_enriched.round(4))

ax = raw_enriched[['nDCG@10', 'Precision@10', 'Recall@20', 'MRR']].plot(
    kind='bar', figsize=(10, 4), rot=0, color=['#697055', '#cf5c52', '#c6a35b', '#60697a']
)
ax.set_ylim(0.5, 1.03)
ax.set_title('Raw vs Enriched 검색 성능')
ax.set_ylabel('score')
plt.show()

print('enriched - raw nDCG@10 = -0.0057')
print('95% CI =', raw_enriched_ci, '→ 0을 포함하므로 통계적으로 뚜렷한 우위 없음')
print('최종 선택: raw — 근소한 수치 우위와 단순성, 원문 의미 보존을 고려')

## 3. Arctic vs KURE 임베딩 모델 비교

동일한 Raw 텍스트, 동일 질의와 라벨을 사용해 한국어 검색 임베딩 모델을 비교했습니다. 정확도뿐 아니라 전체 90만 건 처리에 영향을 주는 임베딩 처리율도 함께 봤습니다.

In [ ]:
embedding_models = pd.DataFrame([
    {'model': 'Arctic', 'nDCG@10': 0.8969, 'Precision@10': 0.9833, 'DirectP@10': 0.8083, 'MRR': 1.0000, 'rows/sec': 58.95},
    {'model': 'KURE',   'nDCG@10': 0.8657, 'Precision@10': 0.9750, 'DirectP@10': 0.7833, 'MRR': 1.0000, 'rows/sec': 56.25},
]).set_index('model')

display(embedding_models.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
embedding_models[['nDCG@10', 'Precision@10', 'DirectP@10', 'MRR']].plot(
    kind='bar', ax=axes[0], rot=0, color=['#697055', '#cf5c52', '#c6a35b', '#60697a']
)
axes[0].set_ylim(0.75, 1.03)
axes[0].set_title('검색 정확도')
embedding_models[['rows/sec']].plot(kind='bar', ax=axes[1], rot=0, color='#697055', legend=False)
axes[1].set_title('문서 임베딩 처리율')
axes[1].set_ylabel('rows/sec')
plt.tight_layout()
plt.show()

print('KURE - Arctic nDCG@10 = -0.0312')
print('95% CI = [-0.0684, 0.0017]')
print('최종 선택: Arctic — 정확도와 처리율 모두 우세')

## 4. ChromaDB vs PostgreSQL+pgvector 비교

두 DB에는 **같은 Arctic 벡터 901,358개**를 적재했습니다. 따라서 이 실험은 임베딩 모델 비교가 아니라 저장·검색 엔진 비교입니다.

- **p50/p95 latency**: 질의 시간의 중앙값/상위 95% 지연
- **Recall@10**: 정확 검색 Top-10 중 근사 검색이 복원한 비율
- **Top-1 일치율**: 정확 검색 1위와 DB 1위가 같은 비율
- 필터 조건: 의회·연도 메타데이터 필터

In [ ]:
db_results = pd.DataFrame([
    {'condition': 'unfiltered', 'database': 'ChromaDB', 'p50_ms': 1.42, 'p95_ms': 2.14, 'Recall@10': 0.87, 'Top1': 0.80},
    {'condition': 'unfiltered', 'database': 'pgvector', 'p50_ms': 2.75, 'p95_ms': 4.31, 'Recall@10': 0.95, 'Top1': 1.00},
    {'condition': 'council+year filter', 'database': 'ChromaDB', 'p50_ms': 586.29, 'p95_ms': 861.08, 'Recall@10': 1.00, 'Top1': 0.90},
    {'condition': 'council+year filter', 'database': 'pgvector', 'p50_ms': 3.97, 'p95_ms': 8.18, 'Recall@10': 1.00, 'Top1': 0.90},
])
display(db_results.round({'p50_ms': 2, 'p95_ms': 2, 'Recall@10': 2, 'Top1': 2}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
latency = db_results.pivot(index='condition', columns='database', values='p50_ms')
latency.plot(kind='bar', ax=axes[0], rot=0, color=['#cf5c52', '#697055'])
axes[0].set_yscale('log')
axes[0].set_title('p50 검색 지연 (로그 축)')
axes[0].set_ylabel('milliseconds')
recall = db_results.pivot(index='condition', columns='database', values='Recall@10')
recall.plot(kind='bar', ax=axes[1], rot=0, color=['#cf5c52', '#697055'])
axes[1].set_ylim(0.8, 1.02)
axes[1].set_title('Recall@10')
plt.tight_layout()
plt.show()

filtered = db_results[db_results['condition'] == 'council+year filter'].set_index('database')
speedup = filtered.loc['ChromaDB', 'p50_ms'] / filtered.loc['pgvector', 'p50_ms']
print(f'필터 검색 p50 기준 pgvector가 약 {speedup:.1f}배 빠름')
print('최종 선택: pgvector — 필터 검색 성능, SQL 결합, 운영 확장성을 고려')

## 5. 예측 평가 지표 구현

예측 과제는 두 문제로 나눴습니다.

1. **다음 달에도 논의되는가?** → 이진 분류 → F1
2. **다음 달 근거 단위가 몇 건인가?** → 개수 회귀 → MAE·RMSE

추가로 전월 대비 증가·유지·감소 방향을 맞혔는지 **방향성 정확도**로 평가합니다. 여기서 ‘언급량’의 단위는 원문 발언 수가 아니라 이슈 군집에 포함된 검색 청크 수입니다.

In [ ]:
def binary_f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'TP': tp, 'FP': fp, 'FN': fn, 'precision': precision, 'recall': recall, 'F1': f1}

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def directional_accuracy(previous, actual, predicted):
    actual_direction = np.sign(np.asarray(actual) - np.asarray(previous))
    predicted_direction = np.sign(np.asarray(predicted) - np.asarray(previous))
    return float(np.mean(actual_direction == predicted_direction))

print(binary_f1([1, 1, 0, 1], [1, 0, 0, 1]))
print('MAE 예시:', mae([10, 5, 8], [8, 7, 9]))
print('RMSE 예시:', round(rmse([10, 5, 8], [8, 7, 9]), 4))

## 6. 시간순 Rolling Backtest

무작위 분할은 미래 데이터를 과거 학습에 섞을 수 있으므로 사용하지 않았습니다. 각 검증 월마다 그보다 이전 월만 학습하고, 검증 월을 한 달씩 앞으로 이동했습니다.

- 대표 이슈: 10개
- 학습 패널: 180행
- 검증 예측: 100건
- Rolling folds: 10개 (2025-03 ~ 2025-12)
- 분류 선택 모델: 최근 활동률 기준선
- 개수 선택 모델: Poisson Regression

In [ ]:
forecast_metrics = pd.DataFrame([
    {'metric': '논의 지속 F1', 'selected_model': 0.8813559322, 'last_month_baseline': 0.8192771084, 'improvement': '+0.0621'},
    {'metric': '언급량 MAE', 'selected_model': 7.1224314605, 'last_month_baseline': 10.1514, 'improvement': '29.83% 감소'},
    {'metric': '언급량 RMSE', 'selected_model': 10.0424688786, 'last_month_baseline': 13.2676, 'improvement': '24.31% 감소'},
    {'metric': '방향성 정확도', 'selected_model': 0.7600, 'last_month_baseline': 0.0200, 'improvement': '+0.7400'},
])
display(forecast_metrics.round({'selected_model': 4, 'last_month_baseline': 4}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
score_rows = forecast_metrics[forecast_metrics['metric'].isin(['논의 지속 F1', '방향성 정확도'])].set_index('metric')
score_rows[['selected_model', 'last_month_baseline']].plot(kind='bar', ax=axes[0], rot=0, color=['#697055', '#d8cdb7'])
axes[0].set_ylim(0, 1.0)
axes[0].set_title('분류·방향성 지표: 높을수록 좋음')
error_rows = forecast_metrics[forecast_metrics['metric'].isin(['언급량 MAE', '언급량 RMSE'])].set_index('metric')
error_rows[['selected_model', 'last_month_baseline']].plot(kind='bar', ax=axes[1], rot=0, color=['#697055', '#d8cdb7'])
axes[1].set_title('개수 예측 오차: 낮을수록 좋음')
plt.tight_layout()
plt.show()

### 예측 결과 해석

- **F1 0.8814**: 지속되는 이슈를 놓치지 않는 Recall과 잘못 지속된다고 판단하는 오류를 억제하는 Precision의 균형이 양호했습니다.
- **MAE 7.12건**: 다음 달 이슈 군집 청크 수 예측이 실제값과 평균 약 7.12건 차이 났습니다.
- **RMSE 10.04건**: 큰 오차에 더 큰 벌점을 주는 지표로, 급증 월에서의 위험까지 확인했습니다.
- **방향성 정확도 0.76**: 증가·유지·감소 방향을 76% 맞혔습니다.
- 전월 값을 그대로 쓰는 단순 기준선보다 MAE 29.83%, RMSE 24.31% 개선했습니다.

> 이 결과는 10개 대표 이슈의 24개월 패널에 대한 프로토타입 평가입니다. 모든 자유검색 이슈에 동일한 일반화 성능을 보장하는 수치로 과장하지 않습니다.

## 7. 최종 선택 요약

| 의사결정 | 비교 대상 | 최종 선택 | 선택 근거 |
|---|---|---|---|
| 임베딩 입력 | Raw vs Enriched | **Raw** | nDCG·Precision·Recall 근소 우위, 구조 단순, 원문 의미 보존 |
| 임베딩 모델 | Arctic vs KURE | **Arctic** | nDCG@10 0.8969, 처리율 58.95행/초 |
| 벡터 DB | ChromaDB vs pgvector | **pgvector** | 필터 p50 3.97ms, SQL·메타데이터 결합 용이 |
| 지속 여부 | 최근 활동률 기준선 등 | **active_ratio_baseline** | Rolling F1 0.8814 |
| 언급량 | 단순 유지·평균·Poisson 등 | **Poisson Regression** | Rolling MAE 7.12, RMSE 10.04 |

최종 운영 파이프라인은 **Raw 텍스트 → Arctic 임베딩 → pgvector 검색 → BM25/RRF → BGE 재순위 → 군집화·타임라인 → 예측**으로 구성했습니다.